# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. Our focus will be on step-by-step programmatic access, data extraction, and typical exploratory data analysis workflows for this dataset.

### Dataset Source
This dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

_All record sets, fields, and columns in this exploration will reference entities by their `@id` as defined in the Croissant schema._

In [ ]:
# Ensure the 'mlcroissant' library is installed
!pip install mlcroissant

## 1. Data Loading

We'll load the metadata and records from the FAIR^2 dataset using `mlcroissant`. This enables us to examine the dataset's contents and schema programmatically.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's review the available record sets in the dataset, and list their `@id`s as well as their fields and corresponding field `@id`s.

_Referencing by `@id` ensures that operations are robust against changes in field/column order or name duplication._

In [ ]:
# List all record sets by @id and show their fields by @id
def list_record_sets_and_fields(ds):
    print("Available Record Sets and Fields (@id):\n")
    record_sets = ds.metadata.record_sets
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}")
        if 'fields' in rs:
            for field in rs['fields']:
                fid = field['@id']
                label = field.get('label', '')
                dtype = field.get('dataType', '')
                print(f"  Field: {fid} | label: {label} | type: {dtype}")
        print()

list_record_sets_and_fields(dataset)


## 3. Data Extraction

We'll extract data from selected record sets into pandas DataFrames for analysis. Make sure to use the record set and field `@id`s gathered from the Data Overview step.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Each record set yields a list of dicts
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for {record_set_id}, shape: {dataframes[record_set_id].shape}")

if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nColumns in first record set ({first_rs}):")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

Let's filter records, normalize a numeric field, and perform some basic grouping and processing. Make sure to reference fields using their `@id`. 

For demonstration, we will choose the first numeric field present in the first record set.

In [ ]:
# Select one record set and a numeric field for demonstration
import numpy as np

# Find a numeric field in the first record set
rs_obj = dataset.metadata.record_sets[0] if dataset.metadata.record_sets else None
first_rs_id = rs_obj['@id'] if rs_obj else None
numeric_field_id = None

if rs_obj is not None and 'fields' in rs_obj:
    for field in rs_obj['fields']:
        dtype = field.get('dataType', '').lower()
        if 'number' in dtype or 'integer' in dtype or 'float' in dtype:
            numeric_field_id = field['@id']
            break

if first_rs_id and numeric_field_id and first_rs_id in dataframes:
    df = dataframes[first_rs_id].copy()
    # Convert numeric column to float for operations
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Remove outliers above/below 3 std, drop NA
    mean = df[numeric_field_id].mean()
    std = df[numeric_field_id].std()
    filtered_df = df[(df[numeric_field_id] > mean - 3 * std) & (df[numeric_field_id] < mean + 3 * std)]
    filtered_df = filtered_df.dropna(subset=[numeric_field_id])
    print(f"Filtered records on 3-sigma for field {numeric_field_id} (type: {df[numeric_field_id].dtype}):")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt grouping by a categorical field if present
    group_field_id = None
    if 'fields' in rs_obj:
        for field in rs_obj['fields']:
            dtype = field.get('dataType', '').lower()
            # Group by first string/text/categorical field different from the numeric one
            if dtype in ['text', 'string'] and field['@id'] != numeric_field_id:
                group_field_id = field['@id']
                break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped.head())
else:
    print("No numeric field found for EDA in the first record set.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field using a histogram and, if applicable, a box plot grouped by the chosen categorical field `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if first_rs_id and numeric_field_id and first_rs_id in dataframes:
    df = dataframes[first_rs_id]
    x = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8,5))
    sns.histplot(x.dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {first_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # Boxplot by categorical field if exists
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: No numeric field detected.")

## 6. Conclusion

In this notebook, we:

- Loaded the FAIR^2 schema-driven dataset using the `mlcroissant` library and explored metadata.
- Programmatically listed all record sets, field `@id`s and mapped the dataset's structure.
- Demonstrated robust extraction of data by record set and field `@id`, yielding DataFrames suitable for analysis.
- Applied exploratory analysis including filtering and normalization of numeric values, referencing fields by `@id` as required by FAIR data standards.
- Visualized the data distributions and groupings for the selected numeric and categorical fields.

**This workflow can be adapted to any Croissant-conformant dataset to ensure reproducible and standards-driven data access and analysis.**